<a href="https://colab.research.google.com/github/SenudaR20/visual-anomaly-detection/blob/main/algorithms/VAE/VAE_hazelnut.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Import Libraries

In [11]:
# Algorithm 2: Variational Autoencoder (VAE)
# Dataset: Hazelnut (MVTec AD)

import os
import random
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf

from PIL import Image

from sklearn.model_selection import train_test_split
from tensorflow.keras import layers, Model
from sklearn.metrics import confusion_matrix, classification_report

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)


In [6]:
# connecting google Drive

from google .colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [9]:
DATA_ROOT = Path(
    "/content/drive/MyDrive/MvTech_Dataset/hazelnut"
)

TRAIN_DIR = DATA_ROOT / "train" / "good"
TEST_DIR = DATA_ROOT / "test"
MASK_DIR = DATA_ROOT / "ground_truth"

assert TRAIN_DIR.exists(), (
    f"Training folder not found: {TRAIN_DIR}"
)

assert TEST_DIR.exists(), (
    f"Test folder not found: {TEST_DIR}"
)

print("Dataset found:", DATA_ROOT)
print("Normal training folder:", TRAIN_DIR)
print("Test folder:", TEST_DIR)


Dataset found: /content/drive/MyDrive/MvTech_Dataset/hazelnut
Normal training folder: /content/drive/MyDrive/MvTech_Dataset/hazelnut/train/good
Test folder: /content/drive/MyDrive/MvTech_Dataset/hazelnut/test


In [10]:

IMAGE_EXTENSIONS = {".png", ".jpg", ".jpeg", ".bmp"}

def get_images(folder):
    folder = Path(folder)

    if not folder.exists():
        return []

    return sorted([
        path
        for path in folder.rglob("*")
        if path.is_file()
        and path.suffix.lower() in IMAGE_EXTENSIONS
    ])


# Normal images for training
train_images = get_images(TRAIN_DIR)

# All test images, including normal and defective examples
test_images = []

for folder in sorted(TEST_DIR.iterdir()):
    if folder.is_dir():
        test_images.extend(get_images(folder))

print("Normal training images:", len(train_images))
print("Total test images:", len(test_images))

print("\nTest categories:")
for folder in sorted(TEST_DIR.iterdir()):
    if folder.is_dir():
        print(f"{folder.name}: {len(get_images(folder))}")


Normal training images: 391
Total test images: 110

Test categories:
crack: 18
cut: 17
good: 40
hole: 18
print: 17


Preprocess the images

In [12]:

IMG_HEIGHT = 128
IMG_WIDTH = 128
CHANNELS = 3

def preprocess_image(image_path):
    image = Image.open(image_path).convert("RGB")

    image = image.resize(
        (IMG_WIDTH, IMG_HEIGHT),
        Image.Resampling.BILINEAR
    )

    return np.asarray(
        image,
        dtype=np.float32
    ) / 255.0


X_train = np.stack([
    preprocess_image(path)
    for path in train_images
])

print("Training array shape:", X_train.shape)
print("Minimum pixel value:", X_train.min())
print("Maximum pixel value:", X_train.max())


Training array shape: (391, 128, 128, 3)
Minimum pixel value: 0.07058824
Maximum pixel value: 1.0
